In [ ]:
import numpy as np
import pandas as pd
from datetime import date, datetime
import calendar
import dai

def main(datasource, start_date, end_date):
    """
    factor function - 使用SQL UDF优化的版本
    
    Args:
        datasource (str): Datasource table name
        start_date (str): Start date in 'YYYY-MM-DD HH:MM:SS' format
        end_date (str): End date in 'YYYY-MM-DD HH:MM:SS' format
    
    Returns:
        pd.DataFrame: Factor data with columns ['date', 'instrument', 'factor']
    """
    import numpy as np
    import pandas as pd
    import dai

    def calculate_mid_price(ask_price1: float, bid_price1: float) -> float:
        """计算 mid_price"""
        if np.isnan(ask_price1):
            return bid_price1
        elif np.isnan(bid_price1):
            return ask_price1
        else:
            return (ask_price1 + bid_price1) / 2

    def calculate_weighted_imbalance(
        bid_volume1: float,
        bid_volume2: float,
        bid_volume3: float,
        bid_volume4: float,
        bid_volume5: float,
        ask_volume1: float,
        ask_volume2: float,
        ask_volume3: float,
        ask_volume4: float,
        ask_volume5: float
    ) -> float:
        """ 计算加权订单簿不平衡度"""
        # 计算衰减权重
        n_levels = 5
        weights = [np.exp(-0.3 * i) for i in range(n_levels)]
        weight_sum = sum(weights)
        weights = [w / weight_sum for w in weights]
        
        # 计算加权买卖成交量
        weighted_bid = (
            bid_volume1 * weights[0] +
            bid_volume2 * weights[1] +
            bid_volume3 * weights[2] +
            bid_volume4 * weights[3] +
            bid_volume5 * weights[4]
        )
        weighted_ask = (
            ask_volume1 * weights[0] +
            ask_volume2 * weights[1] +
            ask_volume3 * weights[2] +
            ask_volume4 * weights[3] +
            ask_volume5 * weights[4]
        )

        # 计算不平衡度
        total = weighted_bid + weighted_ask + 1e-8
        imbalance = (weighted_bid - weighted_ask) / total
        
        return float(imbalance)

    def calculate_relative_spread(ask_price1: float, bid_price1: float, mid_price: float) -> float:
        """计算价差因子"""
        return (ask_price1 - bid_price1) / (mid_price + 1e-8)

    def calculate_depth_ratio(
        ask_volume1: float, ask_volume2: float, ask_volume3: float,
        bid_volume1: float, bid_volume2: float, bid_volume3: float
    ) -> float:
        """深度比率"""
        total_ask = ask_volume1 + ask_volume2 + ask_volume3
        total_bid = bid_volume1 + bid_volume2 + bid_volume3
        return (total_bid - total_ask) / (total_bid + total_ask + 1e-8)

    def label_time_segment(date) -> str:
        """
        为单个日期时间打标签的更高效版本
        """
        hour = date.hour
        minute = date.minute
        total_minutes = hour * 60 + minute
        
        # 09:30-09:44
        if 570 <= total_minutes <= 584:
            return '09:45:00'
        # 09:45-09:59
        elif 585 <= total_minutes <= 599:
            return '10:00:00'
        # 10:00-10:14
        elif 600 <= total_minutes <= 614:
            return '10:15:00'
        # 10:15-10:29
        elif 615 <= total_minutes <= 629:
            return '10:30:00'
        # 10:30-10:44
        elif 630 <= total_minutes <= 644:
            return '10:45:00'
        # 10:45-10:59
        elif 645 <= total_minutes <= 659:
            return '11:00:00'
        # 11:00-11:14
        elif 660 <= total_minutes <= 674:
            return '11:15:00'
        # 11:15-11:30
        elif 675 <= total_minutes <= 690:
            return '11:30:00'
        # 13:00-13:14
        elif 780 <= total_minutes <= 794:
            return '13:15:00'
        # 13:15-13:29
        elif 795 <= total_minutes <= 809:
            return '13:30:00'
        # 13:30-13:44
        elif 810 <= total_minutes <= 824:
            return '13:45:00'
        # 13:45-13:59
        elif 825 <= total_minutes <= 839:
            return '14:00:00'
        # 14:00-14:14
        elif 840 <= total_minutes <= 854:
            return '14:15:00'
        # 14:15-14:29
        elif 855 <= total_minutes <= 869:
            return '14:30:00'
        # 14:30-14:44
        elif 870 <= total_minutes <= 884:
            return '14:45:00'
        # 14:45-14:56
        elif 885 <= total_minutes <= 896:
            return '15:00:00'
        else:
            return 'OTHERS'

    def calculate_factor(raw_pressure: list, mid_prices: list) -> float:
        """计算标准化压力因子"""
        # 首先过滤掉 None 值
        raw_pressure_clean = [x for x in raw_pressure if x is not None]
        mid_prices_clean = [x for x in mid_prices if x is not None]
        
        n = min(len(raw_pressure_clean), len(mid_prices_clean))
        if n < 10:
            return np.nan

        # 转换为 numpy 数组，确保是 float 类型
        raw_pressure_arr = np.array(raw_pressure_clean, dtype=np.float64)
        mid_prices_arr = np.array(mid_prices_clean, dtype=np.float64)
        
        # 检查是否有无效值（inf 或 nan）
        if (np.any(np.isnan(raw_pressure_arr)) or np.any(np.isnan(mid_prices_arr)) or
            np.any(np.isinf(raw_pressure_arr)) or np.any(np.isinf(mid_prices_arr))):
            return np.nan

        # 计算统计量
        raw_pressure_mean = np.mean(raw_pressure_arr)
        raw_pressure_std = np.std(raw_pressure_arr)

        # 计算波动率
        if n > 1:
            try:
                # 确保价格都是正数
                if np.any(mid_prices_arr <= 0):
                    return np.nan
                    
                # 使用向量化计算对数收益率
                log_returns = np.log(mid_prices_arr[1:] / mid_prices_arr[:-1])
                volatility = np.std(log_returns)
            except Exception as e:
                print(f"Error calculating volatility: {e}")
                volatility = 0
        else:
            volatility = 0

        # 标准化原始压力因子
        if raw_pressure_std > 0:
            standardized_pressure = (raw_pressure_arr[-1] - raw_pressure_mean) / raw_pressure_std
        else:
            standardized_pressure = 0
        
        # 根据波动率调整因子
        if n > 10:
            try:
                # 使用numpy计算分位数
                pct_changes = np.abs(np.diff(mid_prices_arr) / mid_prices_arr[:-1])
                volatility_threshold = np.percentile(pct_changes, 80)
            except Exception as e:
                print(f"Error calculating percentile: {e}")
                volatility_threshold = 0.01
        else:
            volatility_threshold = 0.01
        
        if volatility > volatility_threshold:
            adjusted_pressure = standardized_pressure * 0.7
        else:
            adjusted_pressure = standardized_pressure
        
        # 应用非线性变换
        try:
            result = np.tanh(adjusted_pressure)
            # 确保结果是有效的浮点数
            if np.isnan(result) or np.isinf(result):
                return np.nan
            return float(result)
        except Exception as e:
            return np.nan

    sql = f"""
    WITH cte_snapshot AS (
        SELECT
            date, instrument_id, price, volume, 

            -- 交易日
            strftime(date, '%Y-%m-%d') as trading_day,

            -- 计算中间价格
            calculate_mid_price(ask_price1, bid_price1) as mid_price,

            -- 计算 计算加权订单簿不平衡度
            calculate_weighted_imbalance(
                COALESCE(bid_volume1, 0),
                COALESCE(bid_volume2, 0),
                COALESCE(bid_volume3, 0),
                COALESCE(bid_volume4, 0),
                COALESCE(bid_volume5, 0),
                COALESCE(ask_volume1, 0),
                COALESCE(ask_volume2, 0),
                COALESCE(ask_volume3, 0),
                COALESCE(ask_volume4, 0),
                COALESCE(ask_volume5, 0)
            ) as weighted_imbalance,

            -- 计算 价差因子
            calculate_relative_spread(COALESCE(ask_price1, 0), COALESCE(bid_price1, 0), mid_price) as relative_spread,

            -- 计算 深度比率
            calculate_depth_ratio(
                COALESCE(bid_volume1, 0),
                COALESCE(bid_volume2, 0),
                COALESCE(bid_volume3, 0),
                COALESCE(ask_volume1, 0),
                COALESCE(ask_volume2, 0),
                COALESCE(ask_volume3, 0)
            ) as depth_ratio,

            -- 计算 压力因子
            (weighted_imbalance / (sqrt(abs(relative_spread)) + 1e-8)) * weighted_imbalance as raw_pressure,

            -- 给 date 列打标签
            label_time_segment(date) as time_segment

        FROM {datasource}
        WHERE time_segment != 'OTHERS'
    ),
    -- 计算窗口直播
    cte_rolling AS(
    SELECT 
        *,
        lag(mid_price, 1) OVER (PARTITION BY instrument_id, trading_day, time_segment ORDER BY date) as prev_mid_price,
        abs(mid_price / prev_mid_price - 1) as returns,
    FROM cte_snapshot
    ),
    -- 计算分组因子值
    cte_window AS (
        SELECT
            trading_day, time_segment, instrument_id,

            -- raw_pressure, mid_price

            -- 计算压力因子
            -- array_agg: 返回包含列的所有值的 LIST, 请见 DAI 函数列表文档https://bigquant.com/wiki/doc/Rceb2JQBdS
            calculate_factor(array_agg(raw_pressure), array_agg(mid_price)) as factor

        FROM cte_rolling
        GROUP BY instrument_id, trading_day, time_segment
        ORDER BY instrument_id, time_segment
    )
    -- 格式化因子数据
    SELECT
        CAST(CONCAT(f.trading_day, ' ', f.time_segment) AS DATETIME) AS date,
        all_instruments.instrument,
        f.factor
    FROM cte_window f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(
        sql,
        filters={
            'date': [start_date, end_date],
        },
        udf_list=[
            dai.DaiUDF(
                name="calculate_mid_price",
                function=calculate_mid_price,
            ),
            dai.DaiUDF(
                name="calculate_weighted_imbalance",
                function=calculate_weighted_imbalance,
            ),
            dai.DaiUDF(
                name="calculate_relative_spread",
                function=calculate_relative_spread,
            ),
            dai.DaiUDF(
                name="calculate_depth_ratio",
                function=calculate_depth_ratio,
            ),
            dai.DaiUDF(
                name="label_time_segment",
                function=label_time_segment,
            ),
            dai.DaiUDF(
                name="calculate_factor",
                function=calculate_factor,
            ),
        ]
    ).df()

    return df


if __name__ == '__main__':
    """
    FOR Development in __main__

    This section demonstrates how to:
    1. Calculate factors using the main() function
    2. Visualize and analyze results using the factorlens module

    Tips:
        - Adjust the date range to test different time periods
        - Modify the SQL query in main() to create your own factors
        - Use factorlens to evaluate factor performance metrics
    """
    from bigmodule import M
    import structlog

    logger = structlog.get_logger()

    datasource = 'cpt_dwc_2026_stock_hs300_snapshot'
    start_date = '2023-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    logger.info(f"Calculating factor for period: {start_date} to {end_date}")
    data = main(datasource, start_date, end_date)
    logger.info(f"Factor data shape: {data.shape}")
    logger.info(f"\nSample data:\n{data.head()}")

    # 因子评估
    results = M.eval_dwc._latest(
        data=data
    )